# Primera entrega | Datos de Nueva York

**Corte de los datos:** 30 de septiembre de 2026 (hora de Bogotá). **Revisión:** 2 de octubre de 2026.

Este cuaderno se ejecutó en Spark local sobre los extractos incluidos en `datos/`. Cada resultado exploratorio corresponde a esos extractos, ordenados por identificador interno y por tanto no aleatorios. Los cuatro conjuntos obligatorios son arrestos, pobreza, vehículos implicados en colisiones y SAT 2012. El informe SAT 2024 sirve de contexto estatal. El cuaderno no mezcla registros individuales entre las fuentes.

Se documentan el entorno real, diez elementos de exploración, dos gráficas, calidad por campo, transformaciones iniciales y ocho preguntas para la entrega final. Las gráficas se guardan en `figuras/` para que puedan incorporarse al informe y la presentación.

In [2]:
# Cargamos módulos estándar para rutas, metadatos y características del intérprete.
from pathlib import Path
import csv
import json
import platform

# PySpark procesa los JSON Lines con el esquema que infiere a partir de los valores.
from pyspark.sql import SparkSession, functions as F

# Reutilizamos una sesión Spark existente si el entorno del curso ya la proporciona.
spark = SparkSession.builder.appName('NYC_primera_entrega').getOrCreate()
sc = spark.sparkContext

# Informamos únicamente características observadas de ESTA ejecución.
print('Spark:', spark.version)
print('Modo de ejecución:', sc.master)
print('Paralelismo informado por Spark:', sc.defaultParallelism)
print('Java:', spark._jvm.java.lang.System.getProperty('java.version'))
print('Python:', platform.python_version())
print('Memoria del driver configurada:', sc.getConf().get('spark.driver.memory', 'no especificada'))

# Aceptamos la ejecución desde la raíz del repositorio o desde su carpeta padre.
base = Path('datos') if Path('datos/manifest.json').exists() else Path('repo_nyc/datos')
if not (base / 'manifest.json').exists():
    raise FileNotFoundError('No se encontró datos/manifest.json. Abra el repositorio en VS Code.')
project_dir = base.parent

# El manifiesto preserva la consulta original, la fecha y el hash de cada archivo.
manifest = {item['fuente']: item for item in json.loads((base / 'manifest.json').read_text(encoding='utf-8'))}
for name, item in manifest.items():
    count_api = item.get('total_api', [])
    total = count_api[0]['count'] if count_api else 'no disponible'
    print(name, '| extracto:', item['filas_extracto'], '| vista API:', total, '| SHA-256:', item['sha256'])

# Cargamos las cuatro tablas sin alterar los archivos originales.
dfs = {name: spark.read.json(str(base / f'{name}.jsonl')) for name in manifest}


Spark: 4.0.1
Modo de ejecución: local[*]
Paralelismo informado por Spark: 16
Java: 22.0.1
Python: 3.12.14
Memoria del driver configurada: no especificada
arrestos | extracto: 10000 | vista API: 141870 | SHA-256: e132b52fa5fea999558199ab02cd3508215ce17528e27939790e62724a9e7bdd
pobreza | extracto: 3000 | vista API: 68273 | SHA-256: d8daf24ec29885548b83a5f07762b878838b1bd6bba25dd206f1ce396cdb4ae0
vehiculos | extracto: 3000 | vista API: 4551002 | SHA-256: 9ca61bf270aefe095daf05ac78fabedb7e80e33774e34a1d2b059c5c044e180e
educacion | extracto: 478 | vista API: 478 | SHA-256: 709f635962084694f219ca99c3c284ac3c43d16aada0f79a5f2e6abebf66e16e


## 1. Estructura y unidad de análisis

La inspección confirma los campos, tipos inferidos y unidad de cada archivo: arresto, persona, vehículo y escuela. Se muestran dos registros ilustrativos; sus valores no se publican como estadísticas de la ciudad.

In [4]:
# Revisamos las cuatro tablas con el mismo procedimiento para evitar omisiones.
for name, df in dfs.items():
    print('\nFUENTE:', name, '| columnas:', len(df.columns), '| filas del extracto:', df.count())
    # printSchema permite detectar cambios en campos o tipos entre versiones.
    df.printSchema()
    # Dos filas facilitan comprobar formato sin imprimir datos en masa.
    df.show(2, truncate=40)



FUENTE: arrestos | columnas: 19 | filas del extracto: 10000
root
 |-- age_group: string (nullable = true)
 |-- arrest_boro: string (nullable = true)
 |-- arrest_date: string (nullable = true)
 |-- arrest_key: string (nullable = true)
 |-- arrest_precinct: string (nullable = true)
 |-- geocoded_column: struct (nullable = true)
 |    |-- coordinates: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- type: string (nullable = true)
 |-- jurisdiction_code: string (nullable = true)
 |-- ky_cd: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- law_cat_cd: string (nullable = true)
 |-- law_code: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- ofns_desc: string (nullable = true)
 |-- pd_cd: string (nullable = true)
 |-- pd_desc: string (nullable = true)
 |-- perp_race: string (nullable = true)
 |-- perp_sex: string (nullable = true)
 |-- x_coord_cd: string (nullable = true)
 |-- y_coord_cd: string (nullable = true)

## 2. Cobertura temporal y 3. Integridad de claves

Se revisan fechas y claves antes de agrupar. `serialno` identifica la vivienda; la persona se identifica por la combinación `serialno` y `sporder`. Las fechas del extracto no describen toda la cobertura de la vista.

In [6]:
# Definimos las columnas de fecha presentes en los dos archivos de eventos.
date_cols = {'arrestos': 'arrest_date', 'vehiculos': 'crash_date'}
for name, date_col in date_cols.items():
    # El mínimo y máximo revelan el periodo efectivamente descargado.
    dfs[name].select(
        F.min(date_col).alias('fecha_minima'),
        F.max(date_col).alias('fecha_maxima'),
        F.count(F.when(F.col(date_col).isNull(), 1)).alias('fecha_nula'),
    ).show()

# Contrastamos filas con claves distintas y claves ausentes por unidad.
keys = {'arrestos': 'arrest_key', 'vehiculos': 'unique_id', 'educacion': 'dbn'}
for name, key in keys.items():
    dfs[name].agg(
        F.count('*').alias('filas'),
        F.countDistinct(key).alias('claves_distintas'),
        F.count(F.when(F.col(key).isNull(), 1)).alias('claves_nulas'),
    ).show()

# La clave de persona del archivo de pobreza es compuesta.
dfs['pobreza'].agg(
    F.count('*').alias('filas'),
    F.countDistinct(F.struct('serialno', 'sporder')).alias('personas_distintas'),
    F.count(F.when(F.col('serialno').isNull() | F.col('sporder').isNull(), 1)).alias('clave_incompleta'),
).show()


+--------------------+--------------------+----------+
|        fecha_minima|        fecha_maxima|fecha_nula|
+--------------------+--------------------+----------+
|2026-01-01T00:00:...|2026-01-13T00:00:...|         0|
+--------------------+--------------------+----------+

+--------------------+--------------------+----------+
|        fecha_minima|        fecha_maxima|fecha_nula|
+--------------------+--------------------+----------+
|2012-07-01T00:00:...|2017-01-03T00:00:...|         0|
+--------------------+--------------------+----------+

+-----+----------------+------------+
|filas|claves_distintas|claves_nulas|
+-----+----------------+------------+
|10000|           10000|           0|
+-----+----------------+------------+

+-----+----------------+------------+
|filas|claves_distintas|claves_nulas|
+-----+----------------+------------+
| 3000|            3000|           0|
+-----+----------------+------------+

+-----+----------------+------------+
|filas|claves_distintas|clav

## 4. Completitud y 5. Categorías principales

Un nulo, una cadena vacía y una celda suprimida son situaciones distintas. La tabla general cuenta nulos y vacíos; el bloque SAT añade la supresión `s` como ausencia analítica de puntaje.

In [8]:
# Calculamos faltantes por columna; el conteo se refiere solo al extracto.
for name, df in dfs.items():
    missing = [
        F.count(F.when(F.col(col).isNull() | (F.trim(F.col(col).cast('string')) == ''), 1)).alias(col)
        for col in df.columns
    ]
    print('\nFALTANTES:', name)
    df.agg(*missing).show(truncate=False)

# Observamos categorías frecuentes para detectar códigos inesperados.
for name, col in [
    ('arrestos', 'arrest_boro'),
    ('vehiculos', 'vehicle_type'),
    ('vehiculos', 'contributing_factor_1'),
    ('educacion', 'school_name'),
]:
    print('\nCATEGORÍA:', name, col)
    dfs[name].groupBy(col).count().orderBy(F.desc('count')).show(10, truncate=45)



FALTANTES: arrestos
+---------+-----------+-----------+----------+---------------+---------------+-----------------+-----+--------+----------+--------+---------+---------+-----+-------+---------+--------+----------+----------+
|age_group|arrest_boro|arrest_date|arrest_key|arrest_precinct|geocoded_column|jurisdiction_code|ky_cd|latitude|law_cat_cd|law_code|longitude|ofns_desc|pd_cd|pd_desc|perp_race|perp_sex|x_coord_cd|y_coord_cd|
+---------+-----------+-----------+----------+---------------+---------------+-----------------+-----+--------+----------+--------+---------+---------+-----+-------+---------+--------+----------+----------+
|0        |0          |0          |0         |0              |0              |0                |1    |0       |12        |0       |0        |0        |0    |0      |0        |0       |0         |0         |
+---------+-----------+-----------+----------+---------------+---------------+-----------------+-----+--------+----------+--------+---------+---------+

## 6. Geografía y 7. Distribución temporal

La tabla por borough y las series mensuales son diagnósticos del extracto. En particular, el extracto de arrestos comienza en enero y no cubre todo 2026.

In [10]:
# Resumimos arrestos por borough tal como aparece en el archivo de origen.
dfs['arrestos'].groupBy('arrest_boro').count().orderBy(F.desc('count')).show()

# Convertimos las fechas y generamos el mes para cada tabla de eventos.
for name, date_col in date_cols.items():
    events = dfs[name].withColumn('fecha', F.to_date(date_col))
    events = events.withColumn('mes', F.date_format('fecha', 'yyyy-MM'))
    print('\nMESES DEL EXTRACTO:', name)
    events.groupBy('mes').count().orderBy('mes').show(24)


+-----------+-----+
|arrest_boro|count|
+-----------+-----+
|          K| 3092|
|          B| 2335|
|          M| 2199|
|          Q| 2005|
|          S|  369|
+-----------+-----+


MESES DEL EXTRACTO: arrestos
+-------+-----+
|    mes|count|
+-------+-----+
|2026-01|10000|
+-------+-----+


MESES DEL EXTRACTO: vehiculos
+-------+-----+
|    mes|count|
+-------+-----+
|2012-07|   58|
|2012-08|   58|
|2012-09|   66|
|2012-10|   50|
|2012-11|   52|
|2012-12|   72|
|2013-01|   51|
|2013-02|   44|
|2013-03|   55|
|2013-04|   54|
|2013-05|   56|
|2013-06|   52|
|2013-07|   54|
|2013-08|   52|
|2013-09|   64|
|2013-10|   56|
|2013-11|   58|
|2013-12|   68|
|2014-01|   35|
|2014-02|   49|
|2014-03|   55|
|2014-04|   51|
|2014-05|   61|
|2014-06|   70|
+-------+-----+
only showing top 24 rows


## 8. Unidades y duplicación de colisiones

Una fila del conjunto Vehicles es un vehículo. `collision_id` permite reconocer siniestros representados, pero el archivo de siniestros Crashes sería necesario para ubicar geográficamente cada uno y medir lesionados.

In [12]:
# Comparamos filas de vehículos con identificadores de colisión distintos.
vehicles = dfs['vehiculos']
vehicles.agg(
    F.count('*').alias('filas_vehiculos'),
    F.countDistinct('collision_id').alias('colisiones_representadas'),
    F.count(F.when(F.col('collision_id').isNull(), 1)).alias('id_colision_nulo'),
).show()

# La siguiente tabla indica cuántas colisiones del EXTRACTO tienen 1, 2, etc. vehículos registrados.
vehicles.groupBy('collision_id').count().groupBy('count').count().orderBy('count').show(10)


+---------------+------------------------+----------------+
|filas_vehiculos|colisiones_representadas|id_colision_nulo|
+---------------+------------------------+----------------+
|           3000|                    2999|               0|
+---------------+------------------------+----------------+

+-----+-----+
|count|count|
+-----+-----+
|    1| 2998|
|    2|    1|
+-----+-----+



## 9. Pobreza: códigos y pesos

El diccionario oficial adjunto indica que `nycgov_pov_stat` codifica 1 = pobre y 2 = no pobre. `pwgtp` es el peso de persona. No se calcula una tasa de ciudad con este extracto no aleatorio.

In [14]:
# Examinamos la distribución conjunta de borough y estado de pobreza.
poverty = dfs['pobreza']
poverty.groupBy('boro', 'nycgov_pov_stat').count().orderBy('boro', 'nycgov_pov_stat').show(30)

# Revisamos pesos e importes sin convertir el conteo del extracto en una tasa oficial.
poverty.select('pwgtp', 'nycgov_pov_stat', 'nycgov_income', 'nycgov_threshold').describe().show()
print('Interpretación de códigos según el diccionario NYC Opportunity: 1 = pobre, 2 = no pobre.')


+----+---------------+-----+
|boro|nycgov_pov_stat|count|
+----+---------------+-----+
|   1|              1|  109|
|   1|              2|  308|
|   2|              1|  191|
|   2|              2|  998|
|   3|              1|   60|
|   3|              2|  337|
|   4|              1|  146|
|   4|              2|  720|
|   5|              1|   26|
|   5|              2|  105|
+----+---------------+-----+

+-------+-----------------+-------------------+-----------------+------------------+
|summary|            pwgtp|    nycgov_pov_stat|    nycgov_income|  nycgov_threshold|
+-------+-----------------+-------------------+-----------------+------------------+
|  count|             3000|               3000|             3000|              3000|
|   mean|           125.32| 1.8226666666666667|76733.61543773311| 32511.14148799976|
| stddev|93.70736696514965|0.38201422379009375|65444.41895547774|12248.183053734736|
|    min|              100|                  1|            -1146|           16241.6

## 10. SAT: supresión y puntajes convertibles

La letra `s` representa una celda sin puntaje numérico publicado. Se registra como faltante analítico y nunca como cero. Los resultados SAT 2012 no se comparan directamente con el reporte estatal 2024.

In [16]:
# Separamos cadenas numéricas de supresiones o valores no interpretables.
education = dfs['educacion']
sat_columns = [
    'num_of_sat_test_takers',
    'sat_critical_reading_avg_score',
    'sat_math_avg_score',
    'sat_writing_avg_score',
]
for col in sat_columns:
    # La expresión acepta enteros y decimales; los demás valores permanecen nulos.
    numeric = F.when(F.col(col).rlike(r'^\d+(\.\d+)?$'), F.col(col).cast('double'))
    education = education.withColumn(col + '_num', numeric)
    print('\nSAT:', col)
    education.agg(
        F.count(col + '_num').alias('valores_numericos'),
        F.count(F.when(F.lower(F.trim(F.col(col))) == 's', 1)).alias('suprimidos_s'),
        F.count(F.when(F.col(col).isNull(), 1)).alias('nulos_originales'),
    ).show()

# Esta tabla describe solo los puntajes publicados en el extracto escolar.
education.select(*(col + '_num' for col in sat_columns)).describe().show()



SAT: num_of_sat_test_takers
+-----------------+------------+----------------+
|valores_numericos|suprimidos_s|nulos_originales|
+-----------------+------------+----------------+
|              421|          57|               0|
+-----------------+------------+----------------+


SAT: sat_critical_reading_avg_score
+-----------------+------------+----------------+
|valores_numericos|suprimidos_s|nulos_originales|
+-----------------+------------+----------------+
|              421|          57|               0|
+-----------------+------------+----------------+


SAT: sat_math_avg_score
+-----------------+------------+----------------+
|valores_numericos|suprimidos_s|nulos_originales|
+-----------------+------------+----------------+
|              421|          57|               0|
+-----------------+------------+----------------+


SAT: sat_writing_avg_score
+-----------------+------------+----------------+
|valores_numericos|suprimidos_s|nulos_originales|
+-----------------+---------

## Calidad y transformaciones iniciales

Conservamos los datos crudos. Las reglas siguientes crean campos derivados y banderas de validez; no imputan arrestos, siniestros ni puntajes. Se deduplica por la clave correcta de cada unidad y se informa cuántas filas quedan marcadas como válidas.

In [18]:
# Normalizamos fecha y código de borough para los arrestos.
arrests = dfs['arrestos'].withColumn('fecha', F.to_date('arrest_date'))
arrests = arrests.withColumn('boro', F.upper(F.trim('arrest_boro')))
arrests = arrests.withColumn(
    'valida',
    F.col('arrest_key').isNotNull()
    & F.col('fecha').isNotNull()
    & F.col('boro').isin('B', 'K', 'M', 'Q', 'S'),
).dropDuplicates(['arrest_key'])

# Un vehículo válido requiere su clave, la clave de colisión y una fecha interpretable.
vehicles_clean = dfs['vehiculos'].withColumn('fecha', F.to_date('crash_date'))
vehicles_clean = vehicles_clean.withColumn(
    'valida',
    F.col('unique_id').isNotNull()
    & F.col('collision_id').isNotNull()
    & F.col('fecha').isNotNull(),
).dropDuplicates(['unique_id'])

# La persona de pobreza requiere identificador compuesto y peso positivo.
poverty_clean = poverty.withColumn('pwgtp_num', F.col('pwgtp').cast('double'))
poverty_clean = poverty_clean.withColumn(
    'valida',
    F.col('serialno').isNotNull()
    & F.col('sporder').isNotNull()
    & (F.col('pwgtp_num') > 0),
).dropDuplicates(['serialno', 'sporder'])

# La clave escolar debe existir; los puntajes suprimidos siguen siendo nulos analíticos.
education_clean = education.withColumn('valida', F.col('dbn').isNotNull()).dropDuplicates(['dbn'])

# Publicamos la cantidad válida e inválida después de cada transformación.
for name, df in [
    ('arrestos', arrests),
    ('vehiculos', vehicles_clean),
    ('pobreza', poverty_clean),
    ('educacion', education_clean),
]:
    print('\nCALIDAD:', name)
    df.groupBy('valida').count().show()



CALIDAD: arrestos
+------+-----+
|valida|count|
+------+-----+
|  true|10000|
+------+-----+


CALIDAD: vehiculos
+------+-----+
|valida|count|
+------+-----+
|  true| 3000|
+------+-----+


CALIDAD: pobreza
+------+-----+
|valida|count|
+------+-----+
|  true| 3000|
+------+-----+


CALIDAD: educacion
+------+-----+
|valida|count|
+------+-----+
|  true|  478|
+------+-----+



## Dos gráficas descriptivas del extracto

Las imágenes siguientes facilitan leer distribuciones sin presentar el subconjunto como una muestra representativa. Se generan con los recuentos agregados de Spark y se guardan en `figuras/`.

In [20]:
# Matplotlib solo recibe agregados pequeños; Spark sigue procesando los registros.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Creamos una carpeta local para las imágenes reproducibles.
figure_dir = project_dir / 'figuras'
figure_dir.mkdir(exist_ok=True)

# Gráfica 1: composición del extracto de arrestos por borough.
borough_counts = dfs['arrestos'].groupBy('arrest_boro').count().orderBy('arrest_boro').collect()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([row['arrest_boro'] for row in borough_counts], [row['count'] for row in borough_counts], color='#315a74')
ax.set(title='Arrestos por borough en el extracto', xlabel='Código de borough', ylabel='Registros')
fig.tight_layout()
fig.savefig(figure_dir / 'arrestos_borough_extracto.png', dpi=160)
plt.close(fig)

# Gráfica 2: registros de vehículos por año del extracto; no son colisiones de toda NYC.
year_counts = dfs['vehiculos'].withColumn('anio', F.year(F.to_date('crash_date'))).groupBy('anio').count().orderBy('anio').collect()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(row['anio']) for row in year_counts], [row['count'] for row in year_counts], color='#609184')
ax.set(title='Vehículos por año en el extracto', xlabel='Año', ylabel='Registros')
fig.tight_layout()
fig.savefig(figure_dir / 'vehiculos_anio_extracto.png', dpi=160)
plt.close(fig)

# Mostramos rutas y no inferimos tendencias del universo desde estos extractos.
print('Gráficas guardadas en:', figure_dir.resolve())


Gráficas guardadas en: C:\Users\JUAN\.codex\.chatgpt-projects\g-p-6abdd1296d7881918a975189eb328f52\repo_nyc\figuras


![Arrestos por borough en el extracto](figuras/arrestos_borough_extracto.png)

![Vehículos por año en el extracto](figuras/vehiculos_anio_extracto.png)

## Tablas agregadas para entender los extractos

Estas tablas no contestan las preguntas finales de negocio: falta descargar la población completa, incorporar la tabla Crashes, comprobar denominadores y alinear periodos y geografía.

In [23]:
# Arrestos distintos por borough y mes dentro del extracto descargado.
arrests.filter('valida').groupBy(
    'boro', F.date_format('fecha', 'yyyy-MM').alias('mes')
).agg(F.countDistinct('arrest_key').alias('arrestos_extracto')).orderBy('boro', 'mes').show(50)

# Colisiones representadas por mes, evitando contar cada vehículo como otro siniestro.
vehicles_clean.filter('valida').groupBy(
    F.date_format('fecha', 'yyyy-MM').alias('mes')
).agg(F.countDistinct('collision_id').alias('colisiones_representadas_extracto')).orderBy('mes').show(50)


+----+-------+-----------------+
|boro|    mes|arrestos_extracto|
+----+-------+-----------------+
|   B|2026-01|             2335|
|   K|2026-01|             3092|
|   M|2026-01|             2199|
|   Q|2026-01|             2005|
|   S|2026-01|              369|
+----+-------+-----------------+

+-------+---------------------------------+
|    mes|colisiones_representadas_extracto|
+-------+---------------------------------+
|2012-07|                               58|
|2012-08|                               58|
|2012-09|                               66|
|2012-10|                               50|
|2012-11|                               52|
|2012-12|                               72|
|2013-01|                               51|
|2013-02|                               44|
|2013-03|                               55|
|2013-04|                               54|
|2013-05|                               56|
|2013-06|                               52|
|2013-07|                               54

## Significado de los atributos

`Diccionario_atributos.csv` documenta los 111 campos de las cuatro tablas, con tipo, significado y procedencia de la descripción. El archivo incluye el diccionario adjunto oficial de pobreza. `est_eitc` se marca como interpretación provisional porque no aparece descrito individualmente en ese adjunto.

In [25]:
# Leemos el diccionario CSV con la biblioteca estándar, sin alterar los datos originales.
dictionary_path = project_dir / 'Diccionario_atributos.csv'
with dictionary_path.open(encoding='utf-8-sig', newline='') as file:
    dictionary = list(csv.DictReader(file))

# Comprobamos que todos los campos de cada tabla tengan una descripción.
for name, df in dfs.items():
    documented = {row['atributo'] for row in dictionary if row['conjunto'] == name}
    print(name, '| campos:', len(df.columns), '| documentados:', len(documented), '| sin documentar:', sorted(set(df.columns) - documented))


arrestos | campos: 19 | documentados: 19 | sin documentar: []
pobreza | campos: 61 | documentados: 61 | sin documentar: []
vehiculos | campos: 25 | documentados: 25 | sin documentar: []
educacion | campos: 6 | documentados: 6 | sin documentar: []


## Ocho preguntas de negocio para la entrega final

1. ¿Cuáles son las cinco zonas con más arrestos durante un año completo y con geografía validada?
2. ¿Cómo cambia la tasa de arrestos por población entre zonas y periodos comparables?
3. ¿Qué categorías de arresto concentran la mayor variación temporal tras controlar cambios de registro?
4. ¿En qué zonas y horarios se concentran los siniestros con lesiones graves, según la tabla Crashes?
5. ¿Cómo varía la tasa de siniestros según volumen de tránsito o viajes en las zonas estudiadas?
6. ¿Qué factores y tipos de vehículo se reportan con mayor frecuencia en los siniestros de mayor gravedad?
7. ¿Cómo cambia la selección de zonas piloto al revisar contexto de pobreza con estimaciones ponderadas y comparables?
8. ¿Qué efecto tiene un piloto vial o preventivo sobre resultados predefinidos frente a una zona comparable?

Estas preguntas quedan abiertas en esta primera entrega. El SAT 2012 y el informe estatal 2024 contextualizan educación, pero no permiten inferir un efecto individual sobre arrestos o siniestros.

## Siguiente fase

Descargar las vistas completas con fecha de corte, incorporar *Motor Vehicle Collisions – Crashes*, verificar pesos de pobreza y denominadores, y acordar pilotos evaluables. La presente ejecución se realizó en Spark local; los parámetros observados constan en la primera celda de código. Un entorno institucional distinto debe registrar sus propios parámetros.